In [0]:
from pyspark.sql.functions import col, when, lower, year

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

matches = spark.table("workspace.silver.matches_clean")

training_dataset = (
    matches
    .withColumn("year", year(col("date")))
    .withColumn(
        "tournament_weight",
        when(col("tournament") == "FIFA World Cup", 5)
        .when(col("tournament") == "FIFA World Cup qualification", 4)
        .when(col("tournament").isin("UEFA Euro", "Copa América"), 4)
        .when(lower(col("tournament")).contains("qualification"), 3)
        .when(col("tournament") == "Friendly", 1)
        .otherwise(2)
    )
    .withColumn("is_world_cup", when(col("tournament") == "FIFA World Cup", 1).otherwise(0))
    .withColumn("is_world_cup_qualifier", when(col("tournament") == "FIFA World Cup qualification", 1).otherwise(0))
    .withColumn("is_friendly", when(col("tournament") == "Friendly", 1).otherwise(0))
    .withColumn("is_neutral", col("neutral").cast("int"))
    .withColumn("home_win", when(col("result") == "home_win", 1).otherwise(0))
    .withColumn("away_win", when(col("result") == "away_win", 1).otherwise(0))
    .withColumn("draw", when(col("result") == "draw", 1).otherwise(0))
)

training_dataset.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.training_dataset")

print("Tabla Gold training_dataset creada correctamente.")

In [0]:
display(spark.table("workspace.gold.training_dataset").limit(10))

In [0]:
spark.sql("SHOW TABLES IN workspace.gold").show()

In [0]:
spark.table("workspace.gold.training_dataset").groupBy("result").count().show()

In [0]:
import pandas as pd
from collections import defaultdict

# Cargar partidos históricos limpios desde Silver
matches_spark = spark.table("workspace.silver.matches_clean")

df = matches_spark.toPandas()

# Ordenar por fecha para evitar fuga de información
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Función para calcular estadísticas recientes
def calculate_recent_stats(history, n=5):
    recent = history[-n:]
    
    if len(recent) == 0:
        return {
            "recent_games": 0,
            "recent_win_rate": 0,
            "recent_draw_rate": 0,
            "recent_loss_rate": 0,
            "recent_avg_goals_for": 0,
            "recent_avg_goals_against": 0,
            "recent_avg_goal_diff": 0
        }
    
    games = len(recent)
    wins = sum(1 for match in recent if match["result"] == "win")
    draws = sum(1 for match in recent if match["result"] == "draw")
    losses = sum(1 for match in recent if match["result"] == "loss")
    
    goals_for = sum(match["goals_for"] for match in recent)
    goals_against = sum(match["goals_against"] for match in recent)
    
    return {
        "recent_games": games,
        "recent_win_rate": wins / games,
        "recent_draw_rate": draws / games,
        "recent_loss_rate": losses / games,
        "recent_avg_goals_for": goals_for / games,
        "recent_avg_goals_against": goals_against / games,
        "recent_avg_goal_diff": (goals_for - goals_against) / games
    }

# Historial acumulado por selección
team_history = defaultdict(list)

feature_rows = []

for _, row in df.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]
    
    home_score = row["home_score"]
    away_score = row["away_score"]
    
    # Estadísticas antes del partido
    home_stats = calculate_recent_stats(team_history[home_team], n=5)
    away_stats = calculate_recent_stats(team_history[away_team], n=5)
    
    feature_row = row.to_dict()
    
    # Features equipo local
    for key, value in home_stats.items():
        feature_row[f"home_{key}"] = value
    
    # Features equipo visitante
    for key, value in away_stats.items():
        feature_row[f"away_{key}"] = value
    
    # Diferencias entre equipos
    feature_row["recent_win_rate_diff"] = home_stats["recent_win_rate"] - away_stats["recent_win_rate"]
    feature_row["recent_draw_rate_diff"] = home_stats["recent_draw_rate"] - away_stats["recent_draw_rate"]
    feature_row["recent_loss_rate_diff"] = home_stats["recent_loss_rate"] - away_stats["recent_loss_rate"]
    feature_row["recent_avg_goals_for_diff"] = home_stats["recent_avg_goals_for"] - away_stats["recent_avg_goals_for"]
    feature_row["recent_avg_goals_against_diff"] = home_stats["recent_avg_goals_against"] - away_stats["recent_avg_goals_against"]
    feature_row["recent_avg_goal_diff_diff"] = home_stats["recent_avg_goal_diff"] - away_stats["recent_avg_goal_diff"]
    feature_row["recent_games_diff"] = home_stats["recent_games"] - away_stats["recent_games"]
    
    feature_rows.append(feature_row)
    
    # Actualizar historial después del partido
    if home_score > away_score:
        home_result = "win"
        away_result = "loss"
    elif home_score < away_score:
        home_result = "loss"
        away_result = "win"
    else:
        home_result = "draw"
        away_result = "draw"
    
    team_history[home_team].append({
        "result": home_result,
        "goals_for": home_score,
        "goals_against": away_score
    })
    
    team_history[away_team].append({
        "result": away_result,
        "goals_for": away_score,
        "goals_against": home_score
    })

df_features = pd.DataFrame(feature_rows)

display(df_features.head())

In [0]:
# Variables de torneo y contexto
df_features["year"] = df_features["date"].dt.year

def tournament_weight(tournament):
    if tournament == "FIFA World Cup":
        return 5
    elif tournament == "FIFA World Cup qualification":
        return 4
    elif tournament in ["UEFA Euro", "Copa América"]:
        return 4
    elif "qualification" in tournament.lower():
        return 3
    elif tournament == "Friendly":
        return 1
    else:
        return 2

df_features["tournament_weight"] = df_features["tournament"].apply(tournament_weight)

df_features["is_world_cup"] = (df_features["tournament"] == "FIFA World Cup").astype(int)
df_features["is_world_cup_qualifier"] = (df_features["tournament"] == "FIFA World Cup qualification").astype(int)
df_features["is_friendly"] = (df_features["tournament"] == "Friendly").astype(int)
df_features["is_neutral"] = df_features["neutral"].astype(int)

# Variable objetivo numérica para modelos
result_mapping = {
    "home_win": 0,
    "draw": 1,
    "away_win": 2
}

df_features["target"] = df_features["result"].map(result_mapping)

display(df_features.head())

In [0]:
training_features_spark = spark.createDataFrame(df_features)

training_features_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.training_dataset_features")

print("Tabla Gold con features creada correctamente.")

In [0]:
display(spark.table("workspace.gold.training_dataset_features").limit(10))

In [0]:
spark.table("workspace.gold.training_dataset_features").groupBy("result").count().show()

In [0]:
print("Filas:", spark.table("workspace.gold.training_dataset_features").count())